In [ ]:
#Importing the standard libraries
import numpy as np
np.set_printoptions(legacy='1.25',suppress=True)

import matplotlib.pyplot as plt
%matplotlib qt

#Importing the solver modules
from BaF_solver.system import System
from BaF_solver.obe_with_gradient import obe 
from BaF_solver.states import SigmaLevel,PiLevelParity
from BaF_solver.obe_with_gradient import Excitation, Static_Excitation
import time
import warnings
warnings.filterwarnings('ignore')
import datetime

/Users/mangesh/Library/CloudStorage/Box-Box/Density_Matrix/BaF138/sidebands/N_0_137/sinusoidal_driving/experiment_with_cython/src/obe_with_gradient.py:25: UserWarning: Could not detect diffeqpy. Only Python package for OBE solver available.
  warnings.warn(f"Could not detect diffeqpy. Only Python package for OBE solver available.")


In [2]:
import copy

#Reference ordering
Bz = 4604.7
Ez = 0.0
gs_order = [0,1]
es_order = ['1/2-','1/2+']
b=System(gs_order,es_order,B_field = [0.0,0.0,Bz],E_stat_field=[0.0,0.0,Ez],ignore_mF = False)

start = time.perf_counter()

b.sigma_Hamiltonian.generate_bare()
b.sigma_Hamiltonian.Zeeman.generate_Zeeman()
b.sigma_Hamiltonian.Stark.generate_Stark()
b.pi_Hamiltonian.generate_bare()
b.pi_Hamiltonian.Zeeman.generate_Zeeman()
b.pi_Hamiltonian.Stark.generate_Stark()

#Next diagonalize the Hamiltonian for this system
b.sigma_Hamiltonian.diagonalize()
b.pi_Hamiltonian.diagonalize()


G_global = b.sigma_Hamiltonian.diagonalized_states
GH_global = np.round(b.sigma_Hamiltonian.diagonalized_Hamiltonian,6)
E_global =b.pi_Hamiltonian.diagonalized_states
EH_global = np.round(b.pi_Hamiltonian.diagonalized_Hamiltonian,6)

print(f"Bz = {Bz}.")
print(f"Static Hamiltonian took : {time.perf_counter()-start}s. ")

G = copy.copy(G_global)#[3:4+1]
GH = copy.copy(GH_global)#[3:4+1,3:4+1]
E = copy.copy(E_global)
EH = copy.copy(EH_global)



vec_prev_sigma = b.sigma_Hamiltonian.diagonalized_states_as_vectors
vec_prev_pi = b.pi_Hamiltonian.diagonalized_states_as_vectors

sign_organizer_sigma_prev = np.ones(len(G))
sign_organizer_pi_prev = np.ones(len(E))


Bz = 4604.7.
Static Hamiltonian took : 0.29464645800180733s. 


In [3]:
GH_list = [] 
EH_list = [] 
H0_list = []
BR_list = []
Hint_1_list = []
Hstatic_int_list = []
from scipy.linalg import block_diag

Bz_list = np.arange(4604.7,4604.8,0.001)#np.arange(1,31,0.25)
for Bz in Bz_list:
    b=System(gs_order,es_order,B_field = [0.0,0.0,Bz],E_stat_field=[0.0,0.0,Ez],ignore_mF = False)
    start = time.perf_counter()

    b.sigma_Hamiltonian.generate_bare()
    b.sigma_Hamiltonian.Zeeman.generate_Zeeman()
    b.sigma_Hamiltonian.Stark.generate_Stark()
    b.pi_Hamiltonian.generate_bare()
    b.pi_Hamiltonian.Zeeman.generate_Zeeman()
    b.pi_Hamiltonian.Stark.generate_Stark()

    #Next diagonalize the Hamiltonian for this system
    b.sigma_Hamiltonian.diagonalize()
    b.pi_Hamiltonian.diagonalize()

    G_global  = b.sigma_Hamiltonian.diagonalized_states
    GH_global = np.round(b.sigma_Hamiltonian.diagonalized_Hamiltonian,6)
    E_global  = b.pi_Hamiltonian.diagonalized_states
    EH_global = np.round(b.pi_Hamiltonian.diagonalized_Hamiltonian,6)

    #print(f"Bz = {Bz}.")
    #print(f"Static Hamiltonian took : {time.perf_counter()-start}s. ")

    G_temp = G_global#[3:4+1]
    GH_temp = GH_global#[3:4+1,3:4+1]
    
    E_temp = E_global#[0:16]
    EH_temp = EH_global#[0:16,0:16]
    

    vec_current_sigma = b.sigma_Hamiltonian.diagonalized_states_as_vectors
    vec_current_pi = b.pi_Hamiltonian.diagonalized_states_as_vectors

    #check the new ground state's overlap with the the previous ground states    permutation_sigma = []
    vec_current_hermit = vec_current_sigma.conj().T
    permutation_sigma = []



    NG = len(G_temp)
    NE = len(E_temp)
    for i in range(NG):
        #check current ith state ground state
        temp = vec_current_hermit@vec_prev_sigma[:,i]
        max_idx = np.argmax(np.abs(temp))
        permutation_sigma.append(max_idx)
        #ith state in te new list would be the max_idx state


    #check the new excited state's overlap with the the previous excited states
    permutation_pi = []
    sign_organizer_pi = np.zeros_like(sign_organizer_pi_prev)
    vec_current_hermit = vec_current_pi.conj().T
    for j in range(NE):
        #check current jth state excited state
        temp = vec_current_hermit@vec_prev_pi[:,j]
        max_idx = np.argmax(np.abs(temp))
        permutation_pi.append(max_idx)
    
    
    G_new = [G_temp[i] for i in permutation_sigma]    
    E_new = [E_temp[j] for j in permutation_pi]
    GH_new_diag = np.diag(GH_temp)[permutation_sigma]
    EH_new_diag = np.diag(EH_temp)[permutation_pi]
    GH_list.append(GH_new_diag)
    EH_list.append(EH_new_diag)

    H0 = np.diag(np.append(GH_new_diag,EH_new_diag))
    assert np.allclose(np.imag(H0),np.zeros(H0.shape))
    H0_list.append(H0.astype(np.float64))

    b.generate_branching_ratios(G_new,E_new)
    BR = b.branching_ratios
    #BR = np.nan_to_num(BR)
    assert np.allclose(np.imag(BR), np.zeros(BR.shape))

    BR_list.append(BR.astype(np.float64))

    start  = time.perf_counter()
    b.generate_interaction_Hamiltonian(G_new,E_new)
    Hint_1 = b.interaction_Hamiltonian
    


    #Read the operator to express the static electric field in the diagonalized basis
    U_sim_sigma = b.sigma_Hamiltonian.diagonalizing_Matrix
    Hstatic_int_sigma_temp = b.sigma_Hamiltonian.Stark.Z
    Hstatic_int_sigma_temp = U_sim_sigma.conj().T@Hstatic_int_sigma_temp@U_sim_sigma

    #Reorder with permutations
    H_temp_row = np.zeros_like(Hstatic_int_sigma_temp)
    H_temp_col = np.zeros_like(Hstatic_int_sigma_temp)
    for i in range(NG):
        H_temp_row[i,:] = Hstatic_int_sigma_temp[permutation_sigma[i],:]
    
    for i in range(NG):
        H_temp_col[:,i] = H_temp_row[:,permutation_sigma[i]]
    Hstatic_int_sigma = H_temp_col
    #Hstatic_int_sigma = Hstatic_int_sigma[3:4+1, 3:4+1]
    
    
    U_sim_pi = b.pi_Hamiltonian.diagonalizing_Matrix
    Hstatic_int_pi_temp = b.pi_Hamiltonian.Stark.Z
    Hstatic_int_pi_temp = U_sim_pi.conj().T@Hstatic_int_pi_temp@U_sim_pi
    #Reorder with permutations
    H_temp_row = np.zeros_like(Hstatic_int_pi_temp)
    H_temp_col = np.zeros_like(Hstatic_int_pi_temp)
    for i in range(NE):
        H_temp_row[i,:] = Hstatic_int_pi_temp[permutation_pi[i],:]
    
    for i in range(NE):
        H_temp_col[:,i] = H_temp_row[:,permutation_pi[i]]
    Hstatic_int_pi = H_temp_col

    
    Hstatic_int = block_diag(Hstatic_int_sigma, Hstatic_int_pi)


    #print(f'All Hamiltonians took : {time.perf_counter() - start} s.')

    Hint_1_list.append(Hint_1)
    Hstatic_int_list.append(Hstatic_int)

    #rearragne the roder of the the vectors recorded as matrices too
    vec_prev_sigma,vec_prev_pi = vec_current_sigma[:,permutation_sigma],vec_current_pi[:,permutation_pi]
    

In [4]:
import scipy
from scipy.interpolate import interp1d,CubicSpline

# Given data
B_vals = np.arange(4604.7,4604.8,0.001)  # shape (10,)
H0_vals = np.array(H0_list)             # shape (10, 112, 112)

BR_vals = np.array(BR_list)

Hint_1_vals = np.array(Hint_1_list)
Hint1_max = np.amax(np.abs(Hint_1_vals),axis=0)


Hstatic_int_vals = np.array(Hstatic_int_list)
Hstatic_int_max = np.amax(np.abs(Hstatic_int_vals),axis=0)

interpol_kind = 'cubic'

def interpolate(x,y,real_imag = False):
    if real_imag == True:
        y_interp_real = CubicSpline(x, np.real(y),extrapolate= False)#,     kind=interpol_kind, axis=0, bounds_error=False, fill_value='extrapolate')
        y_interp_imag = CubicSpline(x, np.imag(y),extrapolate= False)#,     kind=interpol_kind, axis=0, bounds_error=False, fill_value='extrapolate')
        y_interp = (y_interp_real,y_interp_imag)
    else:
        y_interp = CubicSpline(x, y,extrapolate= False)
    return y_interp



from src.numba_cubicspline import make_fast_vector_spline
def numba_interpolate(x,y,real_imag = False):
    if real_imag == True:
        cs = CubicSpline(x, np.real(y))#,     kind=interpol_kind, axis=0, bounds_error=False, fill_value='extrapolate')
        coefs = np.transpose(cs.c, (1, 2, 0))
        y_interp_real = make_fast_vector_spline(x, coefs)

        cs = CubicSpline(x, np.imag(y))#,     kind=interpol_kind, axis=0, bounds_error=False, fill_value='extrapolate')
        coefs = np.transpose(cs.c, (1, 2, 0))
        y_interp_imag = make_fast_vector_spline(x, coefs)
        y_interp = (y_interp_real,y_interp_imag)
    else:
        cs = CubicSpline(x, y)
        coefs = np.transpose(cs.c, (1, 2, 0))
        y_interp = make_fast_vector_spline(x, coefs)#,     kind=interpol_kind, axis=0, bounds_error=False, fill_value='extrapolate')
    return y_interp


# Flatten for fast interpolation (shape: (10, 112*112))
H0_flat          = H0_vals.reshape(len(B_vals), -1)
BR_flat          = BR_vals.reshape(len(B_vals), -1)
Hint_1_flat      = Hint_1_vals.reshape(len(B_vals), -1)

Hstatic_int_flat = Hstatic_int_vals.reshape(len(B_vals), -1)


H0          = numba_interpolate(B_vals, H0_flat)
BR          = numba_interpolate(B_vals, BR_flat)
Hint_1      = numba_interpolate(B_vals, Hint_1_flat,real_imag = True)


Hstatic_int = numba_interpolate(B_vals, Hstatic_int_flat)


In [5]:
def get_interp_array(A_interp,shape,t,real_imag = False):
    if real_imag:
        real, imag = A_interp
        return (real(t).reshape(shape),imag(t).reshape(shape))
    else:
        return A_interp(t).reshape(shape)


n=len(E)+len(G)
plt.title("Without even parity")
tsigma = 8.192/4
for B in 4604.7433+np.arange(-0.01,0.01,0.001): #+np.array([0.004]):#
    Hint_real,Hint_imag = get_interp_array(Hint_1,(n,n),B,real_imag = True)
    plt.clf()
    #plt.subplot(121)
    plt.imshow(Hint_real,cmap='viridis')
    print(B,Hint_real[3,38])
    #plt.subplot(122)
    #plt.imshow(Hint_imag,cmap='viridis')
    #print(np.abs(my_obe_1.Hint[0][0](T)[7,2])**2 +np.abs(my_obe_1.Hint[0][1](T)[7,2])**2)
    
    plt.pause(0.1)

In [6]:
E[4]

(-0.50325-0j) |J = 0.5-, F1 = 1.0, F = 0.5, mF = -0.5> + 
(0.34868-0j) |J = 0.5-, F1 = 1.0, F = 1.5, mF = -0.5> + 
(0.61917-0j) |J = 0.5-, F1 = 2.0, F = 1.5, mF = -0.5> + 
(-0.49172-0j) |J = 0.5-, F1 = 2.0, F = 2.5, mF = -0.5>

In [7]:
GH[4,4]-EH[4,4]

(-348671505.26774+0j)

In [8]:
GH[3,3]-EH[2,2]

(-348672496.572927+0j)

In [9]:
(GH[4,4]-EH[4,4]) - (GH[3,3]-EH[2,2])

(991.3051869869232+0j)

## Set reference before the run

In [10]:
#Reference ordering
Bz = 4604.74324877138 +10e-4
Ez = 0.0
gs_order = [0,1]
es_order = ['1/2-','1/2+']
b=System(gs_order,es_order,B_field = [0.0,0.0,Bz],E_stat_field=[0.0,0.0,Ez],ignore_mF = False)

start = time.perf_counter()

b.sigma_Hamiltonian.generate_bare()
b.sigma_Hamiltonian.Zeeman.generate_Zeeman()
b.sigma_Hamiltonian.Stark.generate_Stark()
b.pi_Hamiltonian.generate_bare()
b.pi_Hamiltonian.Zeeman.generate_Zeeman()
b.pi_Hamiltonian.Stark.generate_Stark()

#Next diagonalize the Hamiltonian for this system
b.sigma_Hamiltonian.diagonalize()
b.pi_Hamiltonian.diagonalize()


G_global = b.sigma_Hamiltonian.diagonalized_states
GH_global = np.round(b.sigma_Hamiltonian.diagonalized_Hamiltonian,6)
E_global =b.pi_Hamiltonian.diagonalized_states
EH_global = np.round(b.pi_Hamiltonian.diagonalized_Hamiltonian,6)

print(f"Bz = {Bz}.")
print(f"Static Hamiltonian took : {time.perf_counter()-start}s. ")

G = copy.copy(G_global)#[3:4+1]
GH = copy.copy(GH_global)#[3:4+1,3:4+1]
E = copy.copy(E_global)
EH = copy.copy(EH_global)





Bz = 4604.744248771381.
Static Hamiltonian took : 0.03177066706120968s. 


In [11]:
G[3],E[2]

((0.47057-0j) |G = 1.0, N = 0, F1 = 1.0, F = 0.5, mF = -0.5> + 
 (0.67088-0j) |G = 1.0, N = 0, F1 = 1.0, F = 1.5, mF = -0.5> + 
 (-0.35955-0j) |G = 2.0, N = 0, F1 = 2.0, F = 1.5, mF = -0.5> + 
 (-0.44632-0j) |G = 2.0, N = 0, F1 = 2.0, F = 2.5, mF = -0.5>,
 (-0.46539-0j) |J = 0.5-, F1 = 1.0, F = 0.5, mF = 0.5> + 
 (0.64535-0j) |J = 0.5-, F1 = 1.0, F = 1.5, mF = 0.5> + 
 (0.38936-0j) |J = 0.5-, F1 = 2.0, F = 1.5, mF = 0.5> + 
 (-0.46404-0j) |J = 0.5-, F1 = 2.0, F = 2.5, mF = 0.5>)

In [12]:
G[4],E[4]

((-0.5319+0j) |G = 1.0, N = 0, F1 = 1.0, F = 0.5, mF = -0.5> + 
 (0.373+0j) |G = 1.0, N = 0, F1 = 1.0, F = 1.5, mF = -0.5> + 
 (0.59196+0j) |G = 2.0, N = 0, F1 = 2.0, F = 1.5, mF = -0.5> + 
 (-0.477+0j) |G = 2.0, N = 0, F1 = 2.0, F = 2.5, mF = -0.5>,
 (-0.50325+0j) |J = 0.5-, F1 = 1.0, F = 0.5, mF = -0.5> + 
 (0.34868+0j) |J = 0.5-, F1 = 1.0, F = 1.5, mF = -0.5> + 
 (0.61917+0j) |J = 0.5-, F1 = 2.0, F = 1.5, mF = -0.5> + 
 (-0.49172+0j) |J = 0.5-, F1 = 2.0, F = 2.5, mF = -0.5>)

In [13]:
GH[4,4] - EH[4,4]

(-348671505.31481797+0j)

### SIngle run

In [14]:
### Single run
B_offset_range=np.arange(-10,10,1)*1e-4

for sign_sinusoid in [1,-1]:
    r3_list = []
    for B_offset in B_offset_range:
        v_beam = 616e-4 #

        # Unipolar non reversing electric field
        pol = 0
        pos = 0/v_beam
        ##
        #E0 = 1j*1/1000
        E0_NR = 0.212*0.3 #in V/cm
        sig = 1.3/v_beam
        dia = 4*sig
        #E0 = 2*V0/5/1.4;dia = 5*1.40/v_beam
        static_field_APV = Static_Excitation(E0_NR, pol, position = pos, diameter = dia, shape = "Static_Gaussian")

        #A single cycle sinusoid
        
        pol = 0
        pos = 0.0
        ##
        E0_R = sign_sinusoid*0.3 # V/cm
        dia = 5.4/v_beam
        #E0 = 2*V0/5/1.4;dia = 2*1.40/v_beam
        static_field_sinusoidal = Static_Excitation(E0_R, pol, position = pos, diameter = dia, shape = "Sinusoidal")
    

        Gamma = 2.7 #Gamma = 2*np.pi*2.7
        tsigma = 0.1/v_beam
        rabi = 1.0 * Gamma
        pol = 0
        groundState = G[3]
        excitedState = E[2]
        det = 0
        pos = -3.25/v_beam
        dia = 4*tsigma
        temp_field_L1_depletion = Excitation(rabi, pol, groundState, excitedState, detuning = det, position =         pos,  diameter = dia, shape = "Gaussian")
        temp_field_L2_depletion = Excitation(rabi*5 ,   pol, G[4], E[4], detuning = 0.0, position = 3.15/v_beam,  diameter = dia, shape = "Gaussian")

        temp_field_L0_depletion = Excitation(0,    pol, groundState, excitedState, detuning = det, position =  -5.5/v_beam, diameter = dia, shape = "Gaussian")
        temp_field_L3_depletion = Excitation(0 ,   pol, groundState, excitedState, detuning = det, position =   5.5/v_beam, diameter = dia, shape = "Gaussian")




        static_field = [static_field_APV,static_field_sinusoidal]
        optical_fields = [temp_field_L0_depletion,temp_field_L1_depletion,temp_field_L2_depletion,temp_field_L3_depletion]
        B_grad = -0.0135e-3*1

        n=len(E)+len(G)
        Bactual = 4604.74324877138 + B_offset
        print(B_offset)
        H0_val_diag = np.diag(get_interp_array(H0,(n,n),Bactual,real_imag = False))
        #print(H0_val_diag[3])
        BRs = get_interp_array(BR,(len(G),len(E)),Bactual,real_imag = False)
        #print(BRs[3,2])

        steps=50
        
        r_init = np.zeros((n,n),dtype = np.complex128)

        for i in range(len(G)): #Initializing the density matrix considering a rot temperature of 4 K
            if i<4:
                r_init[i,i] = 1.0#/len(G)
            else:
                r_init[i,i] = 1.0#/len(G)



        test_factor = 30
        package = 'Python'
        obe_mode = 'symengine'
        overall_envelope = None
        #print('Creating obes')

        start = time.perf_counter()
        my_obe_1 = obe(optical_fields,[G,E],H0,Hint_1,Hstatic_int,BR,[Bactual,B_grad],static_field, 
                        test_factor, mode = obe_mode, max_Hints = [Hint1_max],max_Hstatic = Hstatic_int_max,overall_envelope = overall_envelope)


        method = 'RK45'
        start = time.perf_counter()
        ans = my_obe_1.solve(steps,r_init,
                            max_step_size = 1/Gamma,#max_step_size,
                            package = package,
                            method = method)
        print(f"Z Solve took {time.perf_counter() - start :.3f}.")
        rho = np.array(ans[-1]) #gives the solution at the end of the time
        r_init = rho.reshape(n,n)
        #print(f"Z solve took {time.time()-start} s.")
        r_diag = np.round(np.diag(r_init),13)
        #print(f"Trace : {np.sum(r_diag[4:24])}")
        #for indx in [0,1,2,3,4,5,6,7]:
        #    print(f"r{indx} = {r_diag[indx].real}",end = ",")
        #print("")

        r3_list.append(r_diag[3].real)
        if sign_sinusoid == 1:
            plus = copy.copy(r3_list)
        elif sign_sinusoid == -1:
            minus = copy.copy(r3_list)
Assym = (np.array(plus)**1 - np.array(minus)**1)/((np.array(plus)**1 + np.array(minus)**1))
plt.plot(2*1.399e3*B_offset_range,Assym,'-o')    

-0.001
Z Solve took 11.391.
-0.0009000000000000001
Z Solve took 6.338.
-0.0008
Z Solve took 8.251.
-0.0007
Z Solve took 5.743.
-0.0006000000000000001
Z Solve took 5.765.
-0.0005
Z Solve took 5.452.
-0.0004
Z Solve took 7.459.
-0.00030000000000000003
Z Solve took 5.910.
-0.0002
Z Solve took 6.618.
-0.0001
Z Solve took 5.998.
0.0
Z Solve took 5.947.
0.0001
Z Solve took 5.423.
0.0002
Z Solve took 5.770.
0.00030000000000000003
Z Solve took 5.947.
0.0004
Z Solve took 5.783.
0.0005
Z Solve took 5.993.
0.0006000000000000001
Z Solve took 5.682.
0.0007
Z Solve took 7.007.
0.0008
Z Solve took 5.689.
0.0009000000000000001
Z Solve took 5.643.
-0.001
Z Solve took 5.481.
-0.0009000000000000001
Z Solve took 5.617.
-0.0008
Z Solve took 5.412.
-0.0007
Z Solve took 5.498.
-0.0006000000000000001
Z Solve took 5.641.
-0.0005
Z Solve took 5.874.
-0.0004
Z Solve took 6.766.
-0.00030000000000000003
Z Solve took 5.751.
-0.0002
Z Solve took 6.520.
-0.0001
Z Solve took 5.805.
0.0
Z Solve took 6.497.
0.0001
Z Sol

In [15]:
Assym = (np.array(plus)**1 - np.array(minus)**1)/((np.array(plus)**1 + np.array(minus)**1))
plt.plot(2*1.399e3*B_offset_range,Assym,'-o')

In [16]:
plt.plot(2*1.399e3*B_offset_range, np.array(plus)**1,'-bo')
plt.plot(2*1.399e3*B_offset_range,np.array(minus)**1,'-ro')


In [17]:
0.0135*2*1.399*1e3

37.773

In [18]:
3.05/v_beam

49.51298701298701

In [19]:
MM = np.genfromtxt("test.csv",delimiter=",")
t = MM[:,0]*v_beam
delt = MM[:,1]*1e3 #in kHz
plt.plot(t,delt)

## Fit the dispersion to extract B

In [20]:
def fitfun(delta,a0,B_GSL,delta_0,x):
    return a0+B_GSL*(delta - delta_0)/((delta - delta_0)**2+x**2)
from scipy.optimize import curve_fit


y_data = Assym
x_data = 2*1.399e3*B_offset_range

p0 = [-0.008,-0.095,0.25,1.1]


plt.plot(x_data,y_data,'b',label = 'simulated data')

idx = np.abs(2*1.399e3*B_offset_range) > 0.1
x_data_for_fit = x_data[idx]
y_data_for_fit = y_data[idx]

idx = x_data_for_fit < 4.0
x_data_for_fit = x_data_for_fit[idx]
y_data_for_fit = y_data_for_fit[idx]


plt.plot(x_data_for_fit,y_data_for_fit,'ob',label = 'stripped data')
popt,pcov = curve_fit(fitfun,x_data_for_fit,y_data_for_fit,p0,maxfev = 20_000)

x_plot = np.linspace(x_data[0],x_data[-1],1000)
plt.plot(x_plot,fitfun(x_plot,*popt),'r',label = 'GSL fit',alpha = 0.7)
print(popt)
plt.legend();
print(popt[1]*1e3)


[-0.   -0.    0.25  1.1 ]
-9.050027985577822e-18


In [21]:
gamma =  2*np.pi*37.6e6*200/616
w=2*np.pi*11.4e3*200/616
sig = 21e-6*616/200
Bgsl = 2/np.sqrt(np.pi)*E0_NR/E0_R*gamma*sig/2*(np.pi**2/2-3-sig**2*w**2/4)
Bgsl/2/np.pi

-129.30630192406326

In [22]:
E0 = [0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
Bsim_Bgsl = np.array([128.31,126.44,123.44,119.65,114.98,109.332,102.31,94.52])/129.31

plt.plot(E0,Bsim_Bgsl,'-ob')

In [23]:
idx = np.abs(2*1.399e3*B_offset_range) > 2.0
idx

array([ True,  True,  True, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
        True,  True])

In [24]:
t = np.linspace(-8.5,8.5,ans.shape[0])*1/v_beam #in us

for i in range(3,5):
    y = []
    for count in range(ans.shape[0]):
        A= ans[count].reshape(n,n)
        pops = np.diag(A).real
        y.append(pops[i])
    plt.plot(t,y,label = str(i))
plt.title(f"With even parity, E_field = {b.E_stat_field[2]}")
plt.grid(True)
plt.legend();

In [25]:
v_beam = 616e-4
def single_sinusoidal(t_symbol,center,width):
    freq = 1/width
    return np.sin(2*np.pi*freq*t_symbol)*pulse_np(t_symbol,center,width)
def pulse_np(t_num,center,width):
    k = 200
    return 0.5 * (np.tanh(k * (t_num-center + width / 2)) - np.tanh(k * (t_num-center - width / 2)))
t = np.linspace(-8.5,8.5,1000)*1/v_beam #in us
y = single_sinusoidal(t,0,5.4/v_beam)
plt.plot(t,y)

In [26]:
3.2*2

6.4

In [27]:
-44.2*v_beam

-2.7227200000000003